#Spark-Based Sales Analysis Project

## 1. Project Overview
This project explores sales, customer behavior, and logistics performance
using Apache Spark DataFrames.

##2. Data Preparation

### 2.1 Data Loading

In [4]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [22]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date
from pyspark.sql.types import DoubleType

spark = SparkSession.builder.appName("University_Project").getOrCreate()

path = "/content/drive/MyDrive/Sparks/train.csv"

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|2017-11-08|2017-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|2017-11-08|2017-11-11|  Second Class|   C

###2.2 Data Frame Definition

In [25]:
# DF1: The Master View
df1 = spark.read.csv(path, header=True, inferSchema=True, quote='"', escape='"', multiLine=True) \
    .withColumn("Order Date", to_date(col("Order Date"), "dd/MM/yyyy")) \
    .withColumn("Ship Date", to_date(col("Ship Date"), "dd/MM/yyyy")) \
    .cache()

# This DF will answer:
# Q1: Product category with highest total sales
# Q2: Top 3 sub-categories from category that has highest total sales

# Display the "Base Table"
df1.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+
|     1|CA-2017-152156|2017-11-08|2017-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset Col...|  261.96|
|     2|CA-2017-152156|2017-11-08|2017-11-11|  Second Class|   C

In [6]:
# DF2 is a child of DF1
df2 = df1.select("Order Date", "Segment", "Sales")

# This DF will answer:
# Q3: AVG sales for consumers
# Q4: Sales evolution for Corporate vs Consumer

# Display the table
df2.show(5)

+----------+---------+--------+
|Order Date|  Segment|   Sales|
+----------+---------+--------+
|2017-11-08| Consumer|  261.96|
|2017-11-08| Consumer|  731.94|
|2017-06-12|Corporate|   14.62|
|2016-10-11| Consumer|957.5775|
|2016-10-11| Consumer|  22.368|
+----------+---------+--------+
only showing top 5 rows


In [11]:
from pyspark.sql.functions import col, dayofweek, when

# 1. Helper Logic: Create the "Calendar View"
df_calendar = df2.select("Order Date").distinct() \
    .withColumn("DayNum", dayofweek(col("Order Date"))) \
    .withColumn("DayType", when(col("DayNum").isin(1, 7), "Weekend").otherwise("Weekday")) \
    .select("Order Date", "DayType")

# 2. Perform the Join (Slide 22)
df3 = df2.join(df_calendar, on="Order Date", how="inner") \
         .select(df2["Order Date"], df2["Sales"], df_calendar["DayType"])

# This DF will answer:
# Q5: Weekend vs Weekday sales comparison

# Display the table
df3.show(5)

+----------+-------+-------+
|Order Date|  Sales|DayType|
+----------+-------+-------+
|2018-05-28| 115.96|Weekday|
|2018-05-28| 13.872|Weekday|
|2018-05-28| 125.13|Weekday|
|2018-05-28|  27.46|Weekday|
|2018-05-28|271.968|Weekday|
+----------+-------+-------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import datediff, udf
from pyspark.sql.types import StringType

# Define UDF Logic (Slide 23)
def delivery_speed(days):
    if days is None: return "Unknown"
    return "Priority" if days <= 2 else "Standard"

speed_udf = udf(delivery_speed, StringType())

# Transformation Lineage
df4 = df1.withColumn("Shipping_Delay", datediff(col("Ship Date"), col("Order Date"))) \
         .withColumn("Service_Level", speed_udf(col("Shipping_Delay"))) \
         .select("City", "Order Date", "Category", "Product Name", "Shipping_Delay", "Service_Level", "Sales")

# This DF will answer:
# Q6: Avg shipment delay per city
# Q7: Most sold product in December

# Display the table
df4.show(5)

+---------------+----------+---------------+--------------------+--------------+-------------+--------+
|           City|Order Date|       Category|        Product Name|Shipping_Delay|Service_Level|   Sales|
+---------------+----------+---------------+--------------------+--------------+-------------+--------+
|      Henderson|2017-11-08|      Furniture|Bush Somerset Col...|             3|     Standard|  261.96|
|      Henderson|2017-11-08|      Furniture|Hon Deluxe Fabric...|             3|     Standard|  731.94|
|    Los Angeles|2017-06-12|Office Supplies|Self-Adhesive Add...|             4|     Standard|   14.62|
|Fort Lauderdale|2016-10-11|      Furniture|Bretford CR4500 S...|             7|     Standard|957.5775|
|Fort Lauderdale|2016-10-11|Office Supplies|Eldon Fold 'N Rol...|             7|     Standard|  22.368|
+---------------+----------+---------------+--------------------+--------------+-------------+--------+
only showing top 5 rows


Assumption: Since all shipments are domestic (US → US), we define "Priority" delivery as completion within 0–2 days, reflecting common premium shipping standards (same-day, next-day, and two-day delivery) in US e-commerce. Deliveries taking 3+ days are labeled as "Standard".



In [18]:
df5 = df1.select("Order ID", "Customer Name", "City", "Category", "Sales")

# This DF will answer:
# Q8: Orders with highest diversity of categories
# Q9: Highest paying customer per city

# Display the table
df5.show(5)

+--------------+---------------+---------------+---------------+--------+
|      Order ID|  Customer Name|           City|       Category|   Sales|
+--------------+---------------+---------------+---------------+--------+
|CA-2017-152156|    Claire Gute|      Henderson|      Furniture|  261.96|
|CA-2017-152156|    Claire Gute|      Henderson|      Furniture|  731.94|
|CA-2017-138688|Darrin Van Huff|    Los Angeles|Office Supplies|   14.62|
|US-2016-108966| Sean O'Donnell|Fort Lauderdale|      Furniture|957.5775|
|US-2016-108966| Sean O'Donnell|Fort Lauderdale|Office Supplies|  22.368|
+--------------+---------------+---------------+---------------+--------+
only showing top 5 rows


In [24]:
from pyspark.sql.functions import avg

df6 = df1.groupBy("Category").agg(avg("Sales").alias("Avg_Sales"))

# This DF will answer:
# Q10: Highest average sales category

# Display the table
df6.show()

+---------------+------------------+
|       Category|         Avg_Sales|
+---------------+------------------+
|Office Supplies|119.38100084616742|
|      Furniture| 350.6537900384984|
|     Technology|  456.401474351901|
+---------------+------------------+



##3. Queries

###3.1 For each city, identify the product category with the highest total sales

In [ ]:
#Code for Q1

###3.2 Identify the top 3 subcategories within the category that has the highest total sales

In [ ]:
#Code for Q2

###3.3 Compute the average sales for orders belonging to the Consumer segment

In [ ]:
#Code for Q3

### 3.4 Analyze the time evolution of sales, grouped by Corporate and Consumer segments

In [ ]:
#Code for Q4

### 3.5 Compare the average daily sales during weekdays with the average sales during weekends using a time series approach

In [ ]:
#Code for Q5

### 3.6 For each city, compute the average shipment delay (in days) between order date and ship date

In [ ]:
#Code for Q6

### 3.7 For each category, determine the most sold product in the month of December

In [ ]:
#Code for Q7

### 3.8 Identify the order that contains the highest number of different product categories

In [ ]:
#Code for Q8

### 3.9 For each city, find the highest-paying customer and rank these customers in descending order of total payment

In [ ]:
#Code for Q9

### 3.10 Calculate the average sales for each category and identify the category with the highest average sales

In [ ]:
#Code for Q10